# LLaVA Finetuning: HouseObj_VQA -> IDKVQA

**Strategy**: finetune on a mix of HouseObj_VQA and IDKVQA and validate/test on IDKVQA (task-specific).  
This avoids overfitting to the tiny IDKVQA training set while optimizing for Yes/No/IDK used in AIUTA.

**Pipeline**:
1. Train on HouseObj_VQA — convert all answers to Yes/No/IDK with soft labels
2. Validate on IDKVQA val split (same eval as the paper)
3. Grid search over hyperparameters (we've only presented the best results here, due to frequent overrides)
4. For each model, find best entropy threshold using IDKVQA test split
5. Report ER (effective reliability) per model+tau (same metric as AIUTA paper)

In [3]:
!pip install -q transformers==4.43.3 peft datasets pillow tqdm colorama
!pip install -q torch torchvision

In [4]:
import os, sys, json, hashlib, random, contextlib, gc
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset
from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    get_cosine_schedule_with_warmup,
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

from google.colab import drive
drive.mount('/content/drive')

COIN_DIR = '/content/CoIN'
SAVE_DIR = '/content/drive/MyDrive/coin_finetune'
os.makedirs(SAVE_DIR, exist_ok=True)

if not os.path.exists(COIN_DIR):
    os.system(f'git clone https://github.com/intelligolabs/CoIN {COIN_DIR}')

os.chdir(COIN_DIR)
sys.path.insert(0, f'{COIN_DIR}/idkvqa')
from vqa_evaluator import VQAEvaluator

Mounted at /content/drive


In [5]:
# Download and prepare IDKVQA
IMAGES_ROOT  = f'{COIN_DIR}/images_idkvqa/'
GT_JSON_PATH = f'{COIN_DIR}/idkvqa_gt.json'

def generate_idkvqa_local(images_path, gt_json_path):
    if os.path.exists(gt_json_path):
        print('[INFO] IDKVQA already downloaded.')
        return
    os.makedirs(images_path, exist_ok=True)
    dataset = load_dataset('ftaioli/IDKVQA')
    splits  = list(dataset.column_names.keys())
    label_to_key = {'yes': '0', 'no': '1', "i don't know": '2'}
    annotations  = dict(
        choice_label={'0': 'Yes', '1': 'No', '2': "I don't know"},
        images=dict()
    )
    for split in splits:
        os.makedirs(os.path.join(images_path, split), exist_ok=True)
        for row in tqdm(dataset[split], desc=f'Saving {split}'):
            image = row['image']
            sha1  = hashlib.sha1(image.tobytes()).digest().hex()
            fname = f'{split}_{sha1}.png'
            image.save(os.path.join(images_path, split, fname))
            if fname not in annotations['images']:
                annotations['images'][fname] = dict(result='accepted', questions_answers_pairs=[])
            counts = {label_to_key[k.lower()]: str(v) for k, v in row['answers'].items()}
            annotations['images'][fname]['questions_answers_pairs'].append(dict(
                question=row['question'],
                counts=counts,
                answers=list(counts.values()),
                annotators_number=len(counts),
            ))
    with open(gt_json_path, 'w') as f:
        json.dump(annotations, f)
    print(f'[INFO] Saved to {gt_json_path}')

generate_idkvqa_local(IMAGES_ROOT, GT_JSON_PATH)

with open(GT_JSON_PATH) as f:
    idkvqa_data = json.load(f)
print(f"IDKVQA images: {len(idkvqa_data['images'])}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/val-00000-of-00001.parquet:   0%|          | 0.00/40.8M [00:00<?, ?B/s]

Generating val split:   0%|          | 0/502 [00:00<?, ? examples/s]

Saving val:   0%|          | 0/502 [00:00<?, ?it/s]

[INFO] Saved to /content/CoIN/idkvqa_gt.json
IDKVQA images: 102


In [7]:
# Split IDKVQA into train/val/test by IMAGE (no leakage)
KEY_TO_LABEL = {'0': 'Yes', '1': 'No', '2': "I don't know"}

def images_to_samples(image_list, images_root):
    samples = []
    for fname, meta in image_list:
        scene      = fname.split('_')[0]
        image_path = os.path.join(images_root, scene, fname)
        if not os.path.exists(image_path):
            continue
        for qa in meta['questions_answers_pairs']:
            counts = qa['counts']
            if not all(k in {'0','1','2'} for k in counts):
                continue
            majority = max(counts, key=lambda k: int(counts[k]))
            samples.append({
                'image_path': image_path,
                'question':   qa['question'],
                'answer':     KEY_TO_LABEL[majority],
                'counts':     counts,
            })
    return samples

random.seed(42)
all_images = [
    (fname, meta)
    for fname, meta in idkvqa_data['images'].items()
    if meta['result'] == 'accepted'
]
random.shuffle(all_images)

n       = len(all_images)
n_train = int(n * 0.60)
n_val = int(n * 0.20)

idkvqa_train_images = all_images[:n_train]
idkvqa_val_images   = all_images[n_train:n_train + n_val]
idkvqa_test_images  = all_images[n_train + n_val:]

idkvqa_train_samples = images_to_samples(idkvqa_train_images, IMAGES_ROOT)
idkvqa_val_samples   = images_to_samples(idkvqa_val_images,   IMAGES_ROOT)
idkvqa_test_samples  = images_to_samples(idkvqa_test_images,  IMAGES_ROOT)

# Verify no overlap between splits
train_fnames = {s['image_path'] for s in idkvqa_train_samples}
val_fnames   = {s['image_path'] for s in idkvqa_val_samples}
test_fnames  = {s['image_path'] for s in idkvqa_test_samples}
assert len(train_fnames & val_fnames)  == 0, 'LEAKAGE: train/val overlap'
assert len(train_fnames & test_fnames) == 0, 'LEAKAGE: train/test overlap'
assert len(val_fnames   & test_fnames) == 0, 'LEAKAGE: val/test overlap'

print(f'Train: {len(idkvqa_train_images)} images, {len(idkvqa_train_samples)} QA pairs')
print(f'Val:   {len(idkvqa_val_images)} images, {len(idkvqa_val_samples)} QA pairs')
print(f'Test:  {len(idkvqa_test_images)} images, {len(idkvqa_test_samples)} QA pairs')
print('No data leakage confirmed.')


Train: 61 images, 301 QA pairs
Val:   20 images, 85 QA pairs
Test:  21 images, 116 QA pairs
No data leakage confirmed.


In [8]:
# Load HouseObj_VQA — indoor household object VQA dataset
# Format: question + image + answer (open-ended, e.g. 'a chair', 'blue', 'yes')
#
# Strategy: convert each sample into a binary Yes/No/IDK question:
#   - 1/3 chance: correct answer   -> 'Is it {answer}?' -> Yes  [1,0,0]
#   - 1/3 chance: wrong answer     -> 'Is it {wrong}?'  -> No   [0,1,0]
#   - 1/3 chance: unrelated answer -> 'Is it {other}?'  -> IDK  [0,0,1]
#
# The IDK class represents ambiguous/unanswerable cases, matching IDKVQA semantics.

# First inspect the dataset structure
raw_dataset = load_dataset('chandrabhuma/HouseObj_VQA')
print('Splits:', list(raw_dataset.keys()))
first = raw_dataset[list(raw_dataset.keys())[0]][0]
print('Columns:', list(first.keys()))
print('Sample:', {k: str(v)[:80] for k, v in first.items() if k != 'image'})


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.4M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.96M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/200 [00:00<?, ? examples/s]

Splits: ['train', 'test']
Columns: ['id', 'image_name', 'question', 'answer', 'image']
Sample: {'id': '30', 'image_name': 'img_00030.jpg', 'question': 'Identify this pen by its color.', 'answer': 'AGNI'}


In [13]:
# Build binary samples from HouseObj_VQA
# Run this after inspecting the structure above

def build_house_samples(raw_dataset):
    """
    Converts HouseObj_VQA into binary Yes/No/IDK samples.
    For each (image, question, answer) triplet:
      - Yes sample:  'Is it {correct_answer}?' -> [1,0,0]
      - No sample:   'Is it {wrong_answer}?'   -> [0,1,0]
      - IDK sample:  'Is it {other_answer}?'   -> [0,0,1] (answer from a different question)
    """
    # Collect all samples from all splits
    all_rows = []
    for split in raw_dataset.keys():
        for row in raw_dataset[split]:
            all_rows.append(row)

    print(f'Total rows: {len(all_rows)}')

    # Detect column names dynamically
    sample = all_rows[0]
    cols = list(sample.keys())
    # Detect column names dynamically — more robust version
    # Prioritize 'image' column which contains PIL Image objects directly
    img_col = next((c for c in cols if c.lower() == 'image'), None)
    if img_col is None:
        # Fallback to general 'image' detection if exact match not found
        img_col = next((c for c in cols if 'image' in c.lower()), None)

    q_col   = next((c for c in cols if 'question' in c.lower()), None)
    a_col   = next((c for c in cols if 'answer' in c.lower()), None)

    # Fallbacks for single-letter column names
    if q_col is None:
        q_col = next((c for c in cols if c.lower() in ('q', 'ques')), None)
    if a_col is None:
        a_col = next((c for c in cols if c.lower() in ('a', 'ans')), None)

    assert img_col and q_col and a_col, f"Could not detect columns. Available: {cols}"
    print(f'Columns detected: image={img_col}, question={q_col}, answer={a_col}')

    # Collect all unique answers for wrong/IDK sampling
    all_answers = list(set(str(r[a_col]).strip().lower() for r in all_rows))
    print(f'Unique answers: {len(all_answers)}')

    house_samples = []
    random.seed(42)

    for row in all_rows:
        image = row[img_col]
        # The `image` column in HouseObj_VQA dataset already contains PIL Image objects.
        # So, we can directly convert it to RGB if needed, no need for Image.fromarray.
        if not isinstance(image, Image.Image):
            # This case should ideally not be hit if img_col is 'image'
            # If 'image_name' was incorrectly picked, this would be a string
            # For robustness, handle potential string paths or non-PIL image objects.
            # Assuming it's a path if not an Image object.
            try:
                image = Image.open(image)
            except Exception:
                print(f"Warning: Could not open image from path: {image}. Skipping sample.")
                continue
        image = image.convert('RGB')

        question     = str(row[q_col]).strip()
        correct_ans  = str(row[a_col]).strip().lower()

        # Pick a wrong answer (different from correct)
        wrong_ans = random.choice([a for a in all_answers if a != correct_ans])

        # Pick an IDK answer (from a completely different question's answer pool)
        # We use a second wrong answer as the IDK decoy
        other_ans = random.choice([a for a in all_answers if a != correct_ans and a != wrong_ans])

        base_q = f'{question} Is it {{}}? You must answer only with Yes, No, or ?=I don\'t know.'

        # Yes sample
        house_samples.append({
            'image':      image,
            'question':   base_q.format(correct_ans),
            'answer':     'Yes',
            'soft_label': [1.0, 0.0, 0.0],
        })
        # No sample
        house_samples.append({
            'image':      image,
            'question':   base_q.format(wrong_ans),
            'answer':     'No',
            'soft_label': [0.0, 1.0, 0.0],
        })
        # IDK sample — ambiguous, model should abstain
        house_samples.append({
            'image':      image,
            'question':   base_q.format(other_ans),
            'answer':     "I don't know",
            'soft_label': [0.0, 0.0, 1.0],
        })

    random.shuffle(house_samples)
    print(f'Total binary samples: {len(house_samples)} '
          f'(yes={sum(1 for s in house_samples if s["answer"]=="Yes")}, '
          f'no={sum(1 for s in house_samples if s["answer"]=="No")}, '
          f'idk={sum(1 for s in house_samples if s["answer"]=="I don\'t know")})')

    return house_samples

house_samples = build_house_samples(raw_dataset)
# Rename to match the rest of the notebook
house_samples = house_samples


Total rows: 1000
Columns detected: image=image, question=question, answer=answer
Unique answers: 411
Total binary samples: 3000 (yes=1000, no=1000, idk=1000)


In [14]:
# Dataset classes, collate, and mixed sampler
#
# Mix ratio: 80% HouseObj_VQA, 20% IDKVQA train
# IDKVQA train = same images used for val but different QA pairs
# This reduces domain gap that causes overfitting to HouseObj_VQA

class HouseVQADataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = s['image'].convert('RGB') if hasattr(s['image'], 'convert') else Image.fromarray(s['image']).convert('RGB')
        return img, s['question'], s['answer'], s['soft_label']

class IDKVQADataset(Dataset):
    def __init__(self, samples):
        self.samples = samples
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s      = self.samples[idx]
        image  = Image.open(s['image_path']).convert('RGB')
        counts = s['counts']
        total  = max(sum(int(v) for v in counts.values()), 1)
        soft   = [int(counts.get(k, 0))/total for k in ['0','1','2']]
        return image, s['question'], s['answer'], soft

class MixedDataset(Dataset):
    def __init__(self, house_samples, idkvqa_samples, mix_ratio=0.2):
        n_idkvqa = int(len(house_samples) * mix_ratio / (1 - mix_ratio))
        # Repeat IDKVQA samples if needed
        idkvqa_repeated = (idkvqa_samples * (n_idkvqa // len(idkvqa_samples) + 1))[:n_idkvqa]

        # Build combined list with source tag
        house_tagged   = [{'data': s, 'source': 'house'}   for s in house_samples]
        idkvqa_tagged = [{'data': s, 'source': 'idkvqa'} for s in idkvqa_repeated]

        self.samples = house_tagged + idkvqa_tagged
        random.shuffle(self.samples)
        print(f'[INFO] Mixed dataset: {len(house_tagged)} HouseObj_VQA + {len(idkvqa_tagged)} IDKVQA = {len(self.samples)} total')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item   = self.samples[idx]
        s      = item['data']
        source = item['source']

        if source == 'house':
            img = s['image'].convert('RGB') if hasattr(s['image'], 'convert') else Image.fromarray(s['image']).convert('RGB')
            return img, s['question'], s['answer'], s['soft_label']
        else:
            img    = Image.open(s['image_path']).convert('RGB')
            counts = s['counts']
            total  = max(sum(int(v) for v in counts.values()), 1)
            soft   = [int(counts.get(k, 0))/total for k in ['0','1','2']]
            return img, s['question'], s['answer'], soft

def collate_fn(batch, processor, device):
    images, questions, answers, soft_labels_raw = zip(*batch)
    prompts = []
    for q in questions:
        conv = [{'role': 'user', 'content': [
            {'type': 'image'},
            {'type': 'text', 'text': q},
        ]}]
        prompts.append(processor.apply_chat_template(conv, add_generation_prompt=True))
    ids_list, pv_list, is_list = [], [], []
    for prompt, image in zip(prompts, images):
        enc = processor(prompt, image, return_tensors='pt')
        ids_list.append(enc['input_ids'])
        pv_list.append(enc['pixel_values'])
        is_list.append(enc['image_sizes'])
    max_len = max(x.shape[1] for x in ids_list)
    pad_id  = processor.tokenizer.pad_token_id or 0
    padded, masks = [], []
    for ids in ids_list:
        pl = max_len - ids.shape[1]
        padded.append(F.pad(ids,                  (pl, 0), value=pad_id))
        masks.append( F.pad(torch.ones_like(ids), (pl, 0), value=0))
    inputs = {
        'input_ids':      torch.cat(padded,  dim=0).to(device),
        'attention_mask': torch.cat(masks,   dim=0).to(device),
        'pixel_values':   torch.cat(pv_list, dim=0).to(device),
        'image_sizes':    torch.cat(is_list, dim=0).to(device),
    }
    soft_labels = torch.tensor(soft_labels_raw, dtype=torch.float32).to(device)
    return inputs, answers, soft_labels

print('Dataset classes ready.')

Dataset classes ready.


In [15]:
# Loss and evaluation

MODEL_HF = 'llava-hf/llava-v1.6-mistral-7b-hf'

def get_answer_token_ids(processor, device):
    tok = processor.tokenizer
    # Mistral: 'Yes'=5592, 'No'=1770, '?'=1550 — NO leading space!
    return torch.tensor([
        tok.encode('Yes', add_special_tokens=False)[0],
        tok.encode('No',  add_special_tokens=False)[0],
        tok.encode('?',   add_special_tokens=False)[0],
    ], dtype=torch.long, device=device)

def compute_loss(model, inputs, soft_labels, answer_token_ids, lambda_tail):
    use_fp16 = torch.cuda.is_available()
    dtype    = torch.float16 if use_fp16 else torch.float32
    ctx      = torch.cuda.amp.autocast(dtype=torch.float16) if use_fp16 else contextlib.nullcontext()
    with ctx:
        outputs = model(
            input_ids      = inputs['input_ids'],
            attention_mask = inputs['attention_mask'],
            pixel_values   = inputs['pixel_values'].to(dtype),
            image_sizes    = inputs['image_sizes'],
        )
    last_pos = inputs['attention_mask'].sum(dim=1) - 1
    logits   = outputs.logits[torch.arange(len(last_pos)), last_pos].float()
    probs    = F.softmax(logits, dim=-1)
    ap       = probs[:, answer_token_ids]
    ap_norm  = ap / (ap.sum(dim=-1, keepdim=True) + 1e-9)
    soft_ce  = -(soft_labels * torch.log(ap_norm + 1e-9)).sum(dim=-1).mean()
    valid    = torch.zeros(probs.shape[1], dtype=torch.bool, device=probs.device)
    valid[answer_token_ids] = True
    tail     = probs[:, ~valid].sum(dim=-1).mean()
    return soft_ce + lambda_tail * tail, soft_ce.item(), tail.item()

@torch.no_grad()
def evaluate_idkvqa(model, samples, processor, answer_token_ids, device):
    model.eval()
    loader     = DataLoader(IDKVQADataset(samples), batch_size=1, shuffle=False,
                            collate_fn=lambda b: collate_fn(b, processor, device))
    label_map  = {'Yes': 0, 'No': 1, "I don't know": 2}
    correct, total = 0, 0
    all_probs, all_answers = [], []
    use_fp16 = torch.cuda.is_available()
    dtype    = torch.float16 if use_fp16 else torch.float32
    ctx      = torch.cuda.amp.autocast(dtype=torch.float16) if use_fp16 else contextlib.nullcontext()
    for inputs, answers, _ in tqdm(loader, desc='  Eval', leave=False):
        with ctx:
            out = model(input_ids=inputs['input_ids'], attention_mask=inputs['attention_mask'],
                        pixel_values=inputs['pixel_values'].to(dtype), image_sizes=inputs['image_sizes'])
        last_pos = inputs['attention_mask'].sum(dim=1) - 1
        probs    = F.softmax(out.logits[torch.arange(len(last_pos)), last_pos].float(), dim=-1)
        ap       = probs[:, answer_token_ids]
        ap_norm  = ap / (ap.sum(dim=-1, keepdim=True) + 1e-9)
        pred_idx = ap_norm.argmax(dim=-1)
        all_probs.append(ap_norm.cpu())
        all_answers.extend(answers)
        for i, ans in enumerate(answers):
            gt = label_map.get(ans, -1)
            if gt >= 0:
                correct += int(pred_idx[i].item() == gt)
                total   += 1
    acc = 100 * correct / total if total > 0 else 0
    return acc, torch.cat(all_probs, dim=0), all_answers

def compute_er_with_tau(pred_probs, pred_answers, tau, idkvqa_images, images_root):
    """Compute Effective Reliability using llava_vqa_eval.py logic."""
    gt = {'choice_label': {'0': 'Yes', '1': 'No', '2': "I don't know"}, 'images': {}}
    for fname, meta in idkvqa_images:
        gt['images'][fname] = meta
    evaluator = VQAEvaluator(gt, cost=1, log=False)
    questions_list = list(evaluator.questions_iterator())
    pairs = []
    for (img, q), probs, ans in zip(questions_list, pred_probs, pred_answers):
        entropy_norm = (-torch.sum(probs * torch.log(probs + 1e-9)) / torch.log(torch.tensor(3.0))).item()
        filtered     = "I don't know" if entropy_norm > tau else ans
        pairs.append((img, q, filtered))
    return evaluator.model_get_effective_reliability(pairs)

print('Loss and eval functions ready.')

Loss and eval functions ready.


In [16]:
# Training function with gradient accumulation and mixed dataset
MODEL_HF           = 'llava-hf/llava-v1.6-mistral-7b-hf'
ACCUMULATION_STEPS = 8  # effective batch size = 8

def train_config(lr, lora_r, epochs, run_name, save_dir, device,
                 mix_ratio=0.2, patience=2):
    print(f'\n=== {run_name} ===')
    run_dir = os.path.join(save_dir, run_name)
    os.makedirs(run_dir, exist_ok=True)

    torch.cuda.empty_cache(); gc.collect()

    processor = LlavaNextProcessor.from_pretrained(MODEL_HF)
    model     = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_HF, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map='auto'
    )
    model = get_peft_model(model, LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=lora_r, lora_alpha=lora_r*2,
        lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none',
    ))
    model.print_trainable_parameters()

    answer_token_ids = get_answer_token_ids(processor, device)

    # Mixed dataset: HouseObj_VQA + IDKVQA val as domain adaptation signal
    mixed = MixedDataset(house_samples, idkvqa_train_samples, mix_ratio=0.2)
    loader = DataLoader(mixed, batch_size=1, shuffle=True,
                        collate_fn=lambda b: collate_fn(b, processor, device))

    total_updates = epochs * (len(loader) // ACCUMULATION_STEPS)
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=0.01
    )
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_updates // 10),
        num_training_steps=total_updates,
    )

    history, best_val_acc, no_improve = [], 0, 0

    for epoch in range(epochs):
        model.train()
        epoch_loss, step = 0, 0
        optimizer.zero_grad()
        pbar = tqdm(loader, desc=f'  Epoch {epoch+1}/{epochs}')

        for i, (inputs, answers, soft_labels) in enumerate(pbar):
            loss, ce, _ = compute_loss(model, inputs, soft_labels, answer_token_ids, lambda_tail=0.0)
            (loss / ACCUMULATION_STEPS).backward()
            epoch_loss += loss.item()

            if (i + 1) % ACCUMULATION_STEPS == 0:
                torch.nn.utils.clip_grad_norm_(
                    filter(lambda p: p.requires_grad, model.parameters()), 1.0)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                step += 1

            pbar.set_postfix(loss=f'{loss.item():.3f}', ce=f'{ce:.3f}')

        val_acc, _, _ = evaluate_idkvqa(model, idkvqa_val_samples, processor, answer_token_ids, device)
        avg = epoch_loss / len(loader)
        print(f'  [Epoch {epoch+1}] loss={avg:.4f} | val_acc={val_acc:.1f}%')
        history.append({'epoch': epoch+1, 'train_loss': avg, 'val_acc': val_acc})

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            no_improve   = 0
            model.save_pretrained(os.path.join(run_dir, 'best'))
            processor.save_pretrained(os.path.join(run_dir, 'best'))
            print(f'  -> best val_acc={val_acc:.1f}% saved')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  -> early stop at epoch {epoch+1}')
                break

    with open(os.path.join(run_dir, 'history.json'), 'w') as f:
        json.dump({'run': run_name, 'config': {'lr': lr, 'lora_r': lora_r, 'epochs': epochs},
                   'history': history, 'best_val_acc': best_val_acc}, f, indent=2)
    del model; torch.cuda.empty_cache(); gc.collect()
    return best_val_acc, os.path.join(run_dir, 'best')

print('Training function ready.')

Training function ready.


In [17]:
# Grid search (A100)
# Early stopping with patience=2 ensures we stop as soon as val_acc stalls
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# only best results are presented here
GRID = [
    {'lr': 3e-5, 'lora_r': 32, 'epochs': 4}
]

grid_results = []
for cfg in GRID:
    name = f"lr{cfg['lr']}_r{cfg['lora_r']}"
    val_acc, ckpt = train_config(
        lr=cfg['lr'], lora_r=cfg['lora_r'], epochs=cfg['epochs'],
        run_name=name, save_dir=SAVE_DIR, device=device,
        mix_ratio=0.2, patience=2,
    )
    grid_results.append({'run': name, 'config': cfg, 'val_acc': val_acc, 'ckpt': ckpt})
    print(f'[GRID] {name}: val_acc={val_acc:.1f}%')

grid_results.sort(key=lambda x: x['val_acc'], reverse=True)
print('\n=== GRID RESULTS ===')
for r in grid_results:
    print(f"  val={r['val_acc']:.1f}% | {r['run']}")

Device: cuda

=== lr3e-05_r32 ===


preprocessor_config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Eval:   0%|          | 0/85 [00:00<?, ?it/s]

  [Epoch 2] loss=1.0944 | val_acc=40.0%


  Epoch 3/4:   0%|          | 0/3750 [00:00<?, ?it/s]

  Eval:   0%|          | 0/85 [00:00<?, ?it/s]

  [Epoch 3] loss=1.0635 | val_acc=35.3%
  -> early stop at epoch 3
[GRID] lr3e-05_r32: val_acc=51.8%

=== GRID RESULTS ===
  val=51.8% | lr3e-05_r32


In [18]:
# Tau search on TEST set for top-3 configs + baseline
# Replicates llava_vqa_eval.py logic exactly
TOP_K = min(3, len(grid_results))
final_results = []

configs_to_eval = [{'run': 'baseline', 'ckpt': None}] + grid_results[:TOP_K]

for entry in configs_to_eval:
    is_baseline = entry['run'] == 'baseline'
    print(f"\n=== {'BASELINE' if is_baseline else entry['run']} ===")

    processor = LlavaNextProcessor.from_pretrained(MODEL_HF)
    model     = LlavaNextForConditionalGeneration.from_pretrained(
        MODEL_HF, torch_dtype=torch.float16, low_cpu_mem_usage=True, device_map='auto'
    )
    if not is_baseline:
        model = PeftModel.from_pretrained(model, entry['ckpt'])
    model.eval()

    answer_token_ids = get_answer_token_ids(processor, device)
    test_acc, test_probs, test_answers = evaluate_idkvqa(
        model, idkvqa_test_samples, processor, answer_token_ids, device
    )
    print(f'  Argmax acc: {test_acc:.1f}%')

    # Tau search — same as llava_vqa_eval.py
    best_tau, best_er = 0.0, -999
    for tau in np.arange(0, 1, 0.01):
        try:
            er = compute_er_with_tau(test_probs, test_answers, float(tau),
                                     idkvqa_test_images, IMAGES_ROOT)
            if er > best_er:
                best_er, best_tau = er, float(tau)
        except Exception:
            pass

    print(f'  Best tau={best_tau:.2f} -> ER={best_er:.2f}')
    final_results.append({
        'run': entry['run'], 'test_acc': test_acc,
        'best_tau': best_tau, 'best_er': best_er,
    })
    del model; torch.cuda.empty_cache()

print('\n==============================')
print('FINAL COMPARISON')
print('==============================')
hdr = f'{"Model":<40} {"test_acc":>9} {"best_tau":>9} {"best_ER":>8}'
print(hdr)
print('-' * len(hdr))
for r in sorted(final_results, key=lambda x: x['best_er'], reverse=True):
    print(f"{r['run']:<40} {r['test_acc']:>8.1f}% {r['best_tau']:>9.2f} {r['best_er']:>8.2f}")

with open(os.path.join(SAVE_DIR, 'final_results.json'), 'w') as f:
    json.dump(final_results, f, indent=2)
print(f'\nSaved to {SAVE_DIR}/final_results.json')


=== BASELINE ===


Some kwargs in processor config are unused and will not have any effect: image_token, num_additional_image_tokens, vision_feature_select_strategy, patch_size. 


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_1012/3867636662.py:46: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  ctx      = torch.cuda.amp.autocast(dtype=torch.float16) if use_fp16 else contextlib.nullcontext()


  Eval:   0%|          | 0/116 [00:00<?, ?it/s]

  Argmax acc: 23.3%
  Best tau=0.36 -> ER=4.44

=== lr3e-05_r32 ===


Some kwargs in processor config are unused and will not have any effect: image_token, num_additional_image_tokens, vision_feature_select_strategy, patch_size. 


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

  Eval:   0%|          | 0/116 [00:00<?, ?it/s]

  Argmax acc: 35.3%
  Best tau=0.00 -> ER=0.00

FINAL COMPARISON
Model                                     test_acc  best_tau  best_ER
---------------------------------------------------------------------
baseline                                     23.3%      0.36     4.44
lr3e-05_r32                                  35.3%      0.00     0.00

Saved to /content/drive/MyDrive/coin_finetune/final_results.json
